# Exemplo de aula — Hexágonos H3 no Databricks

**Disciplina** Engenharia de Dados para IA · **Aula** 4 — Arquiteturas Modernas de Dados (complemento do LAB 4)

**Pergunta de negócio:** *os nossos CDs estão perto de quem a gente atende?* Hoje a LODLog sabe o município de cada cliente, mas "município" é uma unidade ruim para logística: São Paulo tem 1.500 km², e dois clientes no mesmo município podem estar a 40 km um do outro. O H3 troca o município por uma **grade de hexágonos do mesmo tamanho**, e cada hexágono vira uma **chave de junção** comum, como um CEP geográfico.

**O que este notebook mostra (≈25 min, demonstração do professor):**

| Parte | Pergunta | Funções H3 |
|---|---|---|
| 0 | O que é um hexágono H3? | `h3_longlatash3`, `h3_resolution`, `h3_toparent`, `h3_boundaryaswkt` |
| 1 | O endereço do cliente bate com o município declarado? | `h3_longlatash3` + `GROUP BY` |
| 2 | Quais clientes estão no raio de atendimento de cada CD? | `h3_kring` + `JOIN` por igualdade |
| 3 | Cada entrega saiu do CD mais próximo do cliente? | `h3_try_distance` |
| 4 | Mapa das áreas de atendimento | `h3_boundaryasgeojson` |
| 5 | (Opcional) Quais posts do X caem na área de um CD? | `h3_longlatash3` + `JOIN` |

**Pré-requisitos:** seed do LAB 3 (`lodlog_op` e `lodlog_dw` carregados). As funções H3 já vêm no Databricks, sem instalar nada. Segundo a documentação, elas exigem Databricks Runtime 17.0+ e funcionam em todo tipo de computação, **exceto SQL warehouses clássicos**. A célula da Parte 0 serve de teste: se ela rodar, o resto roda.

Referência: [funções geoespaciais H3](https://docs.databricks.com/aws/pt/sql/language-manual/sql-ref-h3-geospatial-functions).

## Parte 0 — O que é um hexágono H3

O H3 (criado pela Uber para preço dinâmico por região) divide o planeta em hexágonos com **16 resoluções**, de 0 (continental) a 15 (≈1 m²). Cada hexágono tem um **ID único** (`BIGINT`, ou texto hexadecimal). Três propriedades importam para engenharia de dados:

1. **Ponto vira chave.** Latitude/longitude viram um ID de hexágono. Perguntas como "estes dois pontos estão perto?" deixam de ser cálculo geométrico e viram `JOIN ... ON a.h3 = b.h3`, que o Spark distribui e otimiza como qualquer junção.
2. **Vizinhos equidistantes.** Num hexágono, os 6 vizinhos estão à mesma distância do centro (num quadrado, a diagonal é mais longe). "Anéis" de vizinhos aproximam bem um raio.
3. **Hierarquia.** Cada hexágono tem um "pai" na resolução de cima. Dá para guardar o dado detalhado e agregar subindo de nível, como um drill-up de cubo OLAP.

| Resolução | Área média | Lado médio | Uso típico |
|---|---|---|---|
| 4 | ≈1.770 km² | ≈26 km | região metropolitana |
| 5 | ≈253 km² | ≈10 km | cidade média |
| 6 | ≈36 km² | ≈3,7 km | zona de cidade grande |
| 7 | ≈5,2 km² | ≈1,4 km | bairro |
| 8 | ≈0,74 km² | ≈0,5 km | algumas quadras |

A célula abaixo indexa o **CD Campinas** em várias resoluções. Repare que o pai na resolução 5 é o mesmo para todas as linhas: é o mesmo lugar visto de mais longe.

In [ ]:
%sql
SELECT r                                                              AS resolucao,
       h3_longlatash3string(-47.0626, -22.9068, r)                   AS h3_texto,
       h3_longlatash3(-47.0626, -22.9068, r)                         AS h3_bigint,
       h3_resolution(h3_longlatash3(-47.0626, -22.9068, r))          AS confere_resolucao,
       h3_h3tostring(h3_toparent(h3_longlatash3(-47.0626, -22.9068, r), 5)) AS pai_resolucao_5,
       left(h3_boundaryaswkt(h3_longlatash3(-47.0626, -22.9068, r)), 70)    AS contorno_wkt
FROM (SELECT explode(sequence(5, 9)) AS r);
-- Atenção à ordem: as funções H3 recebem LONGITUDE primeiro, depois latitude.

## Parte 1 — Indexar CDs e clientes e auditar endereços

Vamos dar a cada CD e a cada cliente o seu hexágono na **resolução 7** (≈ bairro). As coordenadas dos clientes estão em `lodlog_op.endereco_cliente` (sistema operacional); o município declarado está em `lodlog_dw.dim_cliente`.

As duas *views* temporárias abaixo valem até o fim da sessão do notebook.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW h3_cds AS
SELECT sk_cd, codigo, municipio, uf,
       CAST(latitude AS DOUBLE)  AS lat,
       CAST(longitude AS DOUBLE) AS lon,
       h3_longlatash3(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE), 7) AS h3_r7
FROM lodlog_dw.dim_centro_distribuicao;

CREATE OR REPLACE TEMP VIEW h3_clientes AS
SELECT c.sk_cliente, c.razao_social, c.municipio, c.uf,
       CAST(e.latitude AS DOUBLE)  AS lat,
       CAST(e.longitude AS DOUBLE) AS lon,
       h3_longlatash3(CAST(e.longitude AS DOUBLE), CAST(e.latitude AS DOUBLE), 7) AS h3_r7
FROM lodlog_dw.dim_cliente c
JOIN lodlog_op.endereco_cliente e ON e.cliente_id = c.cliente_id
WHERE c.flag_registro_atual;

SELECT municipio, uf, COUNT(*) AS clientes, COUNT(DISTINCT h3_r7) AS hexagonos_r7
FROM h3_clientes
GROUP BY municipio, uf
ORDER BY clientes DESC;

**Auditoria:** se dois municípios diferentes caem no **mesmo hexágono de bairro**, alguém está com a coordenada errada. A consulta abaixo procura hexágonos com mais de um município declarado.

In [ ]:
%sql
SELECT h3_h3tostring(h3_r7)              AS hexagono_r7,
       collect_set(municipio)            AS municipios_declarados,
       COUNT(*)                          AS clientes
FROM h3_clientes
GROUP BY h3_r7
HAVING COUNT(DISTINCT municipio) > 1
ORDER BY clientes DESC;

**O que aparece:** um hexágono de bairro em **São Paulo** reúne clientes declarados em quase todos os municípios da carteira: Curitiba, Salvador, Fortaleza, Rio de Janeiro e outros. Na consulta anterior, de clientes por município/UF, aparecem também pares impossíveis, como "Curitiba/SP" ou "Salvador/RJ". As duas coisas têm a mesma causa: um defeito no seed do LAB 3 que grava a **coordenada padrão de São Paulo** quando não reconhece o par cidade/UF. A explicação completa está na Parte 3, logo depois da consulta do CD mais próximo.

É exatamente o tipo de erro que acontece em produção: a geocodificação falha e o sistema grava um valor padrão em silêncio.

**Para o executivo:** um relatório "entregas por município" mostraria uma carteira espalhada por 11 cidades. Pela coordenada, quase todos esses clientes estão no mesmo bairro de São Paulo. Decisões de rota e de abertura de CD tomadas sobre esse dado herdam o erro. O H3 transforma a checagem "endereço × coordenada" num simples `GROUP BY`.

## Parte 2 — Raio de atendimento de cada CD (k-ring)

`h3_kring(hexagono, k)` devolve o hexágono e todos os vizinhos até **k anéis** de distância. Na **resolução 6** (lado ≈3,7 km, ≈6,4 km entre centros vizinhos), `k = 5` cobre um raio de **≈30 km** em volta do CD.

O truque de engenharia: em vez de calcular a distância de cada cliente a cada CD, a gente **explode** a área do CD em hexágonos e faz um `JOIN` por igualdade com o hexágono do cliente. O hexágono do cliente na resolução 6 sai do da resolução 7 com `h3_toparent`, sem reler a coordenada.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW h3_area_cd AS
SELECT c.sk_cd, c.codigo, explode(h3_kring(h3_toparent(c.h3_r7, 6), 5)) AS h3_r6
FROM h3_cds c;

SELECT a.codigo                                   AS cd,
       COUNT(DISTINCT a.h3_r6)                    AS hexagonos_na_area,
       COUNT(DISTINCT cl.sk_cliente)              AS clientes_no_raio_30km
FROM h3_area_cd a
LEFT JOIN h3_clientes cl ON h3_toparent(cl.h3_r7, 6) = a.h3_r6
GROUP BY a.codigo
ORDER BY clientes_no_raio_30km DESC;

In [ ]:
%sql
-- Clientes FORA do raio de qualquer CD, e o peso deles nas entregas
WITH cobertura AS (SELECT DISTINCT h3_r6 FROM h3_area_cd),
cliente_coberto AS (
  SELECT cl.sk_cliente, cl.municipio, cl.uf, cv.h3_r6 IS NOT NULL AS coberto
  FROM h3_clientes cl
  LEFT JOIN cobertura cv ON cv.h3_r6 = h3_toparent(cl.h3_r7, 6)
)
SELECT cc.municipio, cc.uf, cc.coberto,
       COUNT(DISTINCT cc.sk_cliente)                         AS clientes,
       COUNT(fe.sk_entrega)                                  AS entregas,
       ROUND(100 * COUNT(fe.sk_entrega) / SUM(COUNT(fe.sk_entrega)) OVER (), 1) AS pct_entregas,
       ROUND(100 * AVG(fe.indicador_atraso), 1)              AS pct_atraso
FROM cliente_coberto cc
LEFT JOIN lodlog_dw.fato_entregas fe ON fe.sk_cliente = cc.sk_cliente
GROUP BY cc.municipio, cc.uf, cc.coberto
ORDER BY cc.coberto, entregas DESC;

**Para discutir:** que parcela das entregas vai para clientes a mais de 30 km de qualquer CD (Rio, Goiânia, Brasília, Salvador, Fortaleza)? O atraso deles é maior? Se a resposta for sim, a próxima pergunta é de arquitetura de rede: abrir um CD, contratar um operador local ou renegociar o SLA.

Por causa do defeito descrito na Parte 1, quase todos os clientes caem na área do CD São Paulo. Use os números para praticar a leitura, não para tirar conclusões sobre a rede.

Teste outros raios mudando `k` (e `6` para outra resolução). O resultado muda de forma previsível: é o mesmo dado visto com outra "lente".

## Parte 3 — A entrega saiu do CD mais próximo?

`h3_try_distance(a, b)` conta quantos **passos de hexágono** separam duas células da mesma resolução (devolve `NULL` se não conseguir calcular). Na **resolução 5**, cada passo vale ≈16 km. Para cada cliente, o CD mais próximo é o de menor distância; depois comparamos com o CD que **de fato** despachou cada entrega (`fato_entregas.sk_cd_origem`).

A coluna `km_haversine` (distância em linha reta) entra só para conferência: ela deve ordenar os CDs do mesmo jeito que os anéis.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW h3_cd_mais_proximo AS
WITH pares AS (
  SELECT cl.sk_cliente, cl.municipio, cl.uf, cd.sk_cd, cd.codigo,
         h3_try_distance(h3_toparent(cl.h3_r7, 5), h3_toparent(cd.h3_r7, 5)) AS aneis_r5,
         ROUND(2 * 6371 * asin(sqrt(
             pow(sin(radians(cd.lat - cl.lat) / 2), 2) +
             cos(radians(cl.lat)) * cos(radians(cd.lat)) * pow(sin(radians(cd.lon - cl.lon) / 2), 2))), 0) AS km_haversine
  FROM h3_clientes cl CROSS JOIN h3_cds cd
)
SELECT * FROM pares
QUALIFY ROW_NUMBER() OVER (PARTITION BY sk_cliente ORDER BY aneis_r5 NULLS LAST, km_haversine) = 1;

SELECT municipio, uf, codigo AS cd_mais_proximo, aneis_r5, km_haversine, COUNT(*) AS clientes
FROM h3_cd_mais_proximo
GROUP BY municipio, uf, codigo, aneis_r5, km_haversine
ORDER BY km_haversine;

In [ ]:
%sql
SELECT p.codigo                                                        AS cd_mais_proximo,
       ROUND(100 * AVG(CASE WHEN fe.sk_cd_origem = p.sk_cd THEN 1 ELSE 0 END), 1) AS pct_saiu_do_mais_proximo,
       COUNT(*)                                                        AS entregas
FROM lodlog_dw.fato_entregas fe
JOIN h3_cd_mais_proximo p ON p.sk_cliente = fe.sk_cliente
GROUP BY p.codigo
ORDER BY entregas DESC;

**Atenção: o resultado confirma um defeito no seed do LAB 3, não nas funções H3.** Uma execução real devolveu:

| cd_mais_proximo | pct_saiu_do_mais_proximo | entregas |
|---|---|---|
| CD-SAO-PAULO | 25,0 | 46.198 |
| CD-BELO-HORIZONTE | 15,9 | 2.015 |
| CD-CAMPINAS | 20,8 | 1.506 |
| CD-CURITIBA | 25,1 | 231 |

A coluna `pct_saiu_do_mais_proximo` está certa: cada valor é o peso daquele CD no sorteio do seed. O problema está na coluna `entregas`: **92% delas** (46 mil de ≈50 mil) são de clientes cujo CD mais próximo seria São Paulo, e o CD Porto Alegre nem aparece. A carteira de clientes, que está espalhada por 11 cidades, não justifica isso.

**De onde vem o defeito.** O `lodlog_lab3_seed_v3.py` sorteia a cidade de cada cliente, mas não guarda essa cidade no cadastro. Ao montar o endereço, o código não a encontra e sorteia de novo, **cidade e UF separadamente**:

```python
cidade = cliente["municipio"] if "municipio" in cliente else random.choice(_cidades)[0]
uf     = cliente["uf"]        if "uf" in cliente        else random.choice(_cidades)[1]
```

Por isso aparecem pares impossíveis, como "Curitiba/SP" ou "Salvador/RJ". Como o par não existe na tabela de coordenadas do seed, o código usa a **coordenada padrão, que é a de São Paulo**. Só cerca de 22% dos pares saem válidos, e cerca de 89% dos clientes terminam com a coordenada de São Paulo.

**Por que isso interessa.** Um relatório "entregas por município" nunca mostraria esse erro: cada cliente tem um município declarado, e as contagens parecem normais. Só quando a **coordenada** vira chave (o hexágono H3) o defeito aparece. É o mesmo tipo de falha de geocodificação que acontece em produção: o endereço não é encontrado e o sistema grava um valor padrão em silêncio. Volte à auditoria da Parte 1 e à consulta de clientes por município/UF e procure os sinais desse defeito.

**Para discutir:** em que camada (Bronze, Silver ou Gold) uma regra de qualidade teria barrado esses registros? Que teste simples em SQL detectaria pares cidade/UF impossíveis antes de o dado chegar ao dashboard?

**O que aparece e por quê:** só algo entre 15% e 25% das entregas sai do CD mais próximo, que é apenas o peso de cada CD no sorteio. No seed, o CD de origem é **sorteado** sem olhar para o cliente, então o número é artificial. Numa empresa real, o mesmo número seria um achado sério: frete pago a mais, prazo maior e mais exposição a atraso.

**Para o executivo:** "% de entregas atendidas pelo CD mais próximo" é um KPI de eficiência de rede que o H3 permite calcular com SQL simples, sem ferramenta de GIS.

## Parte 4 — Mapa das áreas de atendimento

`h3_boundaryasgeojson` devolve o contorno de cada hexágono em GeoJSON, que qualquer biblioteca desenha. À esquerda, o Brasil com os clientes agregados pelo **hexágono onde a coordenada caiu** (resolução 4, ≈26 km de lado) (azul: dentro da área de algum CD; cinza: fora). Com o defeito do seed, o ponto de São Paulo concentra quase todos. À direita, um zoom em Campinas/São Paulo, onde dá para ver os hexágonos das áreas de ≈30 km (Parte 2). Só usa `matplotlib`, que já vem no Databricks.

In [ ]:
import json

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

SUPERFICIE, AREA, DENTRO, FORA, TEXTO = "#fcfcfb", "#b7d3f6", "#104281", "#a3a29b", "#52514e"

areas = spark.sql("SELECT codigo, h3_boundaryasgeojson(h3_r6) AS contorno FROM h3_area_cd").toPandas()
cds = spark.sql("SELECT codigo, municipio, lat, lon FROM h3_cds").toPandas()
clientes = spark.sql('''
WITH cobertura AS (SELECT DISTINCT h3_r6 FROM h3_area_cd)
SELECT cl.municipio, cl.lat, cl.lon, h3_toparent(cl.h3_r7, 4) AS h3_r4, cv.h3_r6 IS NOT NULL AS coberto
FROM h3_clientes cl
LEFT JOIN cobertura cv ON cv.h3_r6 = h3_toparent(cl.h3_r7, 6)
''').toPandas()

fig, (ax_br, ax_sp) = plt.subplots(1, 2, figsize=(14, 7.5), facecolor=SUPERFICIE, gridspec_kw={"width_ratios": [1, 1.15]})

# Painel 1 — Brasil: clientes agregados pelo hexágono H3 de resolução 4 (≈26 km de lado) onde a coordenada caiu.
# Agrupar pelo município DECLARADO esconderia o defeito do seed (Parte 3); agrupar pela coordenada o revela.
por_ponto = clientes.groupby("h3_r4").agg(lat=("lat", "mean"), lon=("lon", "mean"), n=("lat", "size"),
                                          coberto=("coberto", "max"), municipio=("municipio", lambda m: m.mode()[0]),
                                          n_municipios=("municipio", "nunique")).reset_index()
for coberto, cor, rotulo in ((True, DENTRO, "Clientes dentro de alguma área"),
                             (False, FORA, "Clientes fora das áreas")):
    sub = por_ponto[por_ponto["coberto"] == coberto]
    ax_br.scatter(sub["lon"], sub["lat"], s=20 + 6 * sub["n"], color=cor, edgecolor=SUPERFICIE, linewidth=1,
                  zorder=3, label=rotulo)
# CD em quadrado vazado: clientes na mesma coordenada continuam visíveis por baixo
ax_br.scatter(cds["lon"], cds["lat"], s=110, marker="s", facecolor="none", edgecolor="#0b0b0b", linewidth=1.5,
              zorder=4, label="CD")
for _, c in por_ponto.iterrows():
    if c["n_municipios"] > 1:
        rotulo = f"{c['n']} clientes de {c['n_municipios']} municípios declarados"
    elif c["municipio"] == "Campinas":  # colado ao ponto de São Paulo; aparece no zoom
        continue
    else:
        rotulo = f"{c['municipio']} · {c['n']}"
    ax_br.annotate(rotulo, (c["lon"], c["lat"]), xytext=(9, 4), textcoords="offset points", fontsize=9, color=TEXTO)
ax_br.set_xlim(-57, -33)
ax_br.set_ylim(-33, -1)
ax_br.legend(loc="lower right", frameon=False, labelcolor=TEXTO, fontsize=9, markerscale=0.6)
ax_br.set_title("Brasil: clientes pelo ponto da coordenada", loc="left", fontsize=11, color=TEXTO)

# Painel 2 — zoom em Campinas / São Paulo: aqui os hexágonos aparecem
for contorno in areas["contorno"]:
    anel = json.loads(contorno)["coordinates"][0]  # [[lon, lat], ...]
    ax_sp.add_patch(Polygon(anel, closed=True, facecolor=AREA, edgecolor=SUPERFICIE, linewidth=0.8))
for coberto, cor in ((True, DENTRO), (False, FORA)):
    sub = clientes[clientes["coberto"] == coberto]
    ax_sp.scatter(sub["lon"], sub["lat"], s=22, color=cor, edgecolor=SUPERFICIE, linewidth=0.6, zorder=3)
ax_sp.scatter(cds["lon"], cds["lat"], s=160, marker="s", facecolor="none", edgecolor="#0b0b0b", linewidth=1.5, zorder=4)
for _, cd in cds.iterrows():
    ax_sp.annotate(cd["codigo"], (cd["lon"], cd["lat"]), xytext=(9, 6), textcoords="offset points", fontsize=9,
                   color="#0b0b0b")
padrao = por_ponto.loc[por_ponto["n"].idxmax()]  # o ponto que concentra a coordenada padrão
ax_sp.annotate(f"{padrao['n']} clientes de {padrao['n_municipios']} municípios declarados\n"
               "no mesmo ponto: coordenada padrão do seed (ver Parte 3)",
               (padrao["lon"], padrao["lat"]), xytext=(-47.7, -23.9), textcoords="data", fontsize=9,
               color=TEXTO, arrowprops={"arrowstyle": "-", "color": TEXTO, "linewidth": 0.6})
ax_sp.set_xlim(-47.75, -45.95)
ax_sp.set_ylim(-23.95, -22.45)
ax_sp.set_title("Zoom Campinas / São Paulo: hexágonos de ≈36 km², 5 anéis por CD", loc="left", fontsize=11, color=TEXTO)

for ax in (ax_br, ax_sp):
    ax.set_facecolor(SUPERFICIE)
    ax.set_aspect("equal")
    ax.tick_params(colors=TEXTO, labelsize=8)
    for lado in ax.spines.values():
        lado.set_color("#d6d5cf")
fig.suptitle("Áreas de atendimento de ≈30 km por CD (H3, resolução 6, k = 5)", x=0.02, ha="left",
             fontsize=13, color="#0b0b0b")
plt.tight_layout()
plt.show()

## Parte 5 (opcional) — Posts do X perto de um CD

Só funciona se você rodou o `LAB4-X-API.ipynb`, que cria `lodlog_lake.x_eventos_silver_api` com latitude/longitude dos posts. No LAB 4-X a geolocalização usa retângulo + polígono (`shapely`). Com H3, "este post está perto de qual CD?" vira o mesmo `JOIN` por igualdade da Parte 2, sem nenhuma biblioteca de geometria.

In [ ]:
%sql
SELECT a.codigo AS cd_afetado, p.tipo_evento, p.data_evento, p.cidade, p.origem_coord, COUNT(*) AS posts
FROM lodlog_lake.x_eventos_silver_api p
JOIN h3_area_cd a ON a.h3_r6 = h3_longlatash3(p.longitude, p.latitude, 6)
WHERE p.tipo_evento <> 'Irrelevante' AND p.latitude IS NOT NULL
GROUP BY a.codigo, p.tipo_evento, p.data_evento, p.cidade, p.origem_coord
ORDER BY p.data_evento DESC;

## Para fechar a discussão

1. **Resolução é decisão de negócio.** Resolução 6 com `k = 5` dá ≈30 km. Que raio faz sentido para entrega no mesmo dia? E para caminhão de 15 t? Quem decide isso: o engenheiro de dados, o cientista ou a operação?
2. **Por que guardar o H3 como coluna na Silver/Gold?** Calcular uma vez e reutilizar em todo `JOIN` geográfico (clientes, telemetria, posts, clima) é o que torna a análise espacial barata em escala. Em qual camada vocês colocariam?
3. **Hexágono não é município.** O hexágono não respeita fronteira administrativa. Quando o relatório **precisa** de município (imposto, regulação), use a malha do IBGE; quando a pergunta é de **distância e densidade**, use H3.